# Round 2: Naive Bayes (Multinomial & Complement)

**Goal**: Implement Naive Bayes baselines.
- **Multinomial NB**: Classic baseline, fast but overconfident.
- **Complement NB**: Designed for imbalanced datasets.

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import json
import os
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB, ComplementNB
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix

sns.set(style='whitegrid')
reports_dir = 'round2/reports'
os.makedirs(reports_dir, exist_ok=True)
print("Libraries imported.")

## 1. Load Data

In [ ]:
train_df = pd.read_excel('round2/train.xlsx')
val_df = pd.read_excel('round2/val.xlsx')

X_train = train_df['cleaned_poem'].astype(str)
X_val = val_df['cleaned_poem'].astype(str)

# Primary Labels
y_train_p = train_df['primary_id']
y_val_p = val_df['primary_id']

# Secondary Labels
y_train_s = train_df['secondary_id']
y_val_s = val_df['secondary_id']

# Load Maps
with open('round2/label_maps.json', 'r') as f:
    maps = json.load(f)
p_map = {v: k for k, v in maps['primary_map'].items()}
s_map = {v: k for k, v in maps['secondary_map'].items()}

## 2. Feature Extraction (Word-level TF-IDF for NB)
NB usually works better with word counts/tfidf than char n-grams.

In [ ]:
vectorizer = TfidfVectorizer(analyzer='word', stop_words='english', max_features=10000)
X_train_vec = vectorizer.fit_transform(X_train)
X_val_vec = vectorizer.transform(X_val)
print(f"Feature Shape: {X_train_vec.shape}")

## 3. Train Helper

In [ ]:
def train_evaluate_nb(model_cls, y_train, y_val, label_map, prefix):
    print(f"\nTraining {prefix}...")
    model = model_cls()
    model.fit(X_train_vec, y_train)
    
    y_pred = model.predict(X_val_vec)
    
    acc = accuracy_score(y_val, y_pred)
    macro = f1_score(y_val, y_pred, average='macro')
    
    print(f"Accuracy: {acc:.4f}")
    print(f"Macro F1: {macro:.4f}")
    
    # Save Metrics
    metrics = {'accuracy': acc, 'macro_f1': macro}
    with open(f'{reports_dir}/{prefix}_metrics.json', 'w') as f:
        json.dump(metrics, f, indent=4)
        
    # Confusion Matrix
    cm = confusion_matrix(y_val, y_pred)
    plt.figure(figsize=(12, 10))
    sns.heatmap(cm, cmap='Oranges', fmt='d')
    plt.title(f'{prefix} Confusion Matrix')
    plt.savefig(f'{reports_dir}/{prefix}_confusion_matrix.png')
    plt.show()
    
    # Save Errors
    res = pd.DataFrame({'True': y_val, 'Pred': y_pred})
    res['True_Label'] = res['True'].map(label_map)
    res['Pred_Label'] = res['Pred'].map(label_map)
    errors = res[res['True'] != res['Pred']]
    errors.to_csv(f'{reports_dir}/{prefix}_errors.csv', index=False)

## 4. Run Multinomial NB

In [ ]:
# Primary
train_evaluate_nb(MultinomialNB, y_train_p, y_val_p, p_map, 'mnb_primary')
# Secondary
train_evaluate_nb(MultinomialNB, y_train_s, y_val_s, s_map, 'mnb_secondary')

## 5. Run Complement NB

In [ ]:
# Primary
train_evaluate_nb(ComplementNB, y_train_p, y_val_p, p_map, 'cnb_primary')
# Secondary
train_evaluate_nb(ComplementNB, y_train_s, y_val_s, s_map, 'cnb_secondary')